# Study on Reflection pattern of agentic programming

## Prepare

In [1]:
import langchain
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import SystemMessage, HumanMessage

print(f'Import LangChain V{langchain.__version__}')

Import LangChain V1.2.7


In [2]:
with open('api-key/deepseek.txt', 'r') as f:
    llm = ChatOpenAI(
        base_url="https://api.deepseek.com",
        api_key=f.read().strip(),
        model="deepseek-v4-flash",
        temperature=0,
    )

# print(
#     StrOutputParser().invoke(
#         llm.invoke([("human", "Introduce yourself briefly")])
#     )
# )

## Reflection loop

In [3]:
def run_reflection(task_prompt: str, reflect_prompt: str, success_key: str,
                   max_step: int = 3) -> tuple[int, bool, str]:
    message_history = [HumanMessage(content=task_prompt)]
    generated = ""
    is_suc = False
    for i in range(max_step):
        print("="*8 + f"Task {i + 1}th Iteration" + "="*8)
        # generate
        if i == 0:
            print(f">>> Phase 1: Generate content...")
        else:
            print(f">>> Phase 1: Optimize content due to critique...")
            message_history.append(HumanMessage(content="Optimize due to critique."))
        response = llm.invoke(message_history)
        message_history.append(response)
        generated = response.content
        # print(f">>> Generated (V{i+1}):")
        # print(generated)
        # print()
        # reflection
        print(">>> Phase 2: Reflect generated content...")
        critique_response = llm.invoke([
            SystemMessage(content=reflect_prompt),
            HumanMessage(
                content=f"Origin task:\n{task_prompt}\n\n Review content:\n{generated}")
        ])
        critique = critique_response.content
        # examine
        if success_key in critique:
            print("Success!")
            is_suc = True
            break
        # print(f">>> Critique:")
        # print(critique)
        print()
        message_history.append(HumanMessage(content=f"Last critique: {critique}"))
    return (i+1, is_suc, generated)

## Demo from book

In [4]:
# prompt of main task
task_prompt = """你的任务是创建一个名为 `calculate_factorial` 的 Python 函数，满足：
- 只接受一个整数参数 n
- 计算其阶乘 (n!)
- 包含清晰的 docstring, 说明函数功能
- 处理边界情况 (0 的阶乘为 1)
- 处理无效输入 (若输入为负数则抛出 ValueError)
"""

# success key
success_key = "CODE_IS_PERFECT"

# prompt of reflection
reflect_prompt = f"""你是一名资深软件工程师，精通 Python。
你的职责是对提供的 Python 代码进行细致代码审查。
请根据原始任务要求，严格评估代码。
检查是否有 bug、风格问题、遗漏边界情况及其他可改进之处。
若代码完美且满足所有要求，仅回复 '{success_key}'。
否则，请以项目符号列表形式给出批判意见。
"""

In [5]:
step, is_succeed, code = run_reflection(
    task_prompt, reflect_prompt, success_key)

print(f"Run {step} rounds, result: {is_succeed}")
print("Final generated:")
print(code)

========Task 1th Iteration========
>>> Phase 1: Generate content...
>>> Phase 2: Reflect generated content...
Success!
Run 1 rounds, result: True
Final generated:
下面是一个符合要求的 `calculate_factorial` 函数实现：

```python
def calculate_factorial(n: int) -> int:
    """
    计算非负整数 n 的阶乘 (n!).
    
    阶乘的定义为:
        n! = n * (n-1) * (n-2) * ... * 1
    其中 0! = 1.
    
    参数:
        n (int): 要计算阶乘的非负整数。
    
    返回:
        int: n 的阶乘结果。
    
    抛出:
        ValueError: 如果输入为负数。
    
    示例:
        >>> calculate_factorial(5)
        120
        >>> calculate_factorial(0)
        1
        >>> calculate_factorial(-3)
        Traceback (most recent call last):
            ...
        ValueError: 输入必须是非负整数，收到 -3
    """
    if not isinstance(n, int):
        raise TypeError("输入必须是整数")
    if n < 0:
        raise ValueError(f"输入必须是非负整数，收到 {n}")
    
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result
```

此函数：
- 使用循环迭代计算阶乘，避免递归深度限制。
- 包含详细的 docstring，说明功能、参数、返回值和异常。

## Try another job

In [6]:
write_story = """作为一个魔幻现实主义作家，编写一个1000～1500字范围的短片小说，建议：
- 主人公在年轻的时候，一次梦到自己死去，游历死后世界发现非常无聊，和死神表达了想回去的念头，然后就醒了；
- 多年来，她一直觉得那只是一个梦，直到她真的死去，却在那个特别的早晨醒来；
- 此后，她经历了很多次人生，每次都在死后回到那个早晨，她被困在了这段时间里；
- 她不断的尝试，大胆或荒谬，似乎没有任何条件可以触发真正的死亡；
- 故事的最后，她慢慢习惯了，毫无顾忌的尝试各种喜欢的事情，也渐渐明白，即使真的摆脱了，自己也永远不可能知道摆脱，而且每个人其实都生活在这种不知生死的状态中。"""

good_story = "GOOD_STORY"

review_story = """作为一个小说领域优秀的编辑，对收到的故事进行审查，给予修改意见。
当认为已经完美时，仅回复 '{good_story}'。"""

In [7]:
rounds, is_good, story = run_reflection(
    write_story, review_story, good_story)

print(f"Run {rounds} rounds, result: {is_good}")
print("Final generated:")
print(story)

========Task 1th Iteration========
>>> Phase 1: Generate content...
>>> Phase 2: Reflect generated content...

========Task 2th Iteration========
>>> Phase 1: Optimize content due to critique...
>>> Phase 2: Reflect generated content...

========Task 3th Iteration========
>>> Phase 1: Optimize content due to critique...
>>> Phase 2: Reflect generated content...

Run 3 rounds, result: False
Final generated:
她第一次死的时候是二十三岁。那是一个春天的梦。

死后世界什么都没有——一片白雾，一条河，一个坐在船头削苹果的摆渡人。河对岸是间灰白色的大厅，塑料椅子稀稀落落，几个灵魂坐着发呆，像等一辆永远不会来的车。她等了很久，问摆渡人接下来干什么，摆渡人把削好的苹果递给她。

她咬了一口，什么味道都没有。

“这里好无聊，”她说，“我想回去。”

摆渡人看了她一眼：“你确定？”

“确定。”

然后她就醒了。清晨六点十七分，窗外的光灰白色，像极了死后世界的颜色。她发了会儿呆，翻个身，起床，上班，日子照常走。

之后三十多年，她结婚、离婚、生子、旅行，早把这个梦忘得一干二净。五十七岁那年冬天，心脏病发，她倒在自家客厅的地板上，听见心脏像拧紧的弹簧突然崩断。然后她睁开眼，看见摆渡人坐在床沿削苹果。闹钟显示六点十七分，是她二十三岁租住的那间公寓。

她猛地坐起来，大口喘气。摆渡人把苹果递过来，她没接。

“这不是梦，”她说，“我死过一次了。”

摆渡人没说话。

从那天起她开始各种尝试。起初她非常小心：足不出户，吃最健康的食物，把家里所有尖锐的、易碎的、可能致死的东西全部扔掉。但她发现，即使她不主动寻死，五十七岁那年的心脏病也像铁轨尽头一样必然出现。她试过自杀——跳楼，割腕，吞安眠药，把自己关进装满一氧化碳的车库。死亡体验各不相同，结果却完全一致：闭眼，再睁眼，二十三岁，六点十七